In [36]:
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

df = pd.read_csv("../Prajituri.csv")

df["descriere"] = (
    "Denumire: " + df["denumire"].astype(str) +
    " | Ingrediente: " + df["ingrediente"].astype(str) +
    " | Ingredient de baza: " + df["Ingredient de baza"].astype(str) +
    " | Textura: " + df["Textura"].astype(str) +
    " | Dulceata: " + df["Dulceata"].astype(str) +
    " | Provenienta: " + df["Provenienta"].astype(str)
)

# Încarcă DistilBERT multilingv
checkpoint = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:,0,:].squeeze().numpy()

# Exemplu preferință user
user_pref = ("Vreau ceva dulce cu mango")

# Embedding pentru preferință
user_vec = get_embedding(user_pref)

# Embeddings pentru prăjituri (ex: ingrediente + denumire)
cakes_texts = df["descriere"]
cakes_vecs = [get_embedding(t) for t in cakes_texts]


# Similaritate cosinus
sims = cosine_similarity([user_vec], cakes_vecs)[0]

# Top 3 recomandari
# Similarități
sims = cosine_similarity([user_vec], cakes_vecs)[0]

# Top 3 recomandari
top_idx = np.argsort(sims)[::-1][:3]
for i in top_idx:
    print(df["denumire"].iloc[i], "-> scor similaritate:", round(sims[i], 3))



Fructul Padurii -> scor similaritate: 0.961
Monoportie Capodopera -> scor similaritate: 0.96
Eros Dubai -> scor similaritate: 0.96


In [33]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

df = pd.read_csv("../Prajituri.csv")
df["descriere"] = (
    "Denumire: " + df["denumire"].astype(str) +
    " | Ingrediente: " + df["ingrediente"].astype(str) +
    " | Ingredient de baza: " + df["Ingredient de baza"].astype(str) +
    " | Textura: " + df["Textura"].astype(str) +
    " | Dulceata: " + df["Dulceata"].astype(str) +
    " | Provenienta: " + df["Provenienta"].astype(str)
)


#modelul multilingual pentru similaritate
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#embeddinguri pentru prajituri
cakes_vecs = model.encode(df["descriere"].tolist(), convert_to_tensor=True)

def recomanda_prajituri(user_input, top_k=3):
    user_vec = model.encode(user_input, convert_to_tensor=True)
    cos_scores = util.cos_sim(user_vec, cakes_vecs)[0]
    top_idx = np.argsort(cos_scores.cpu().numpy())[::-1][:top_k]

    rezultate = []
    for i in top_idx:
        rezultate.append({
            "denumire": df["denumire"].iloc[i],
            "scor": round(cos_scores[i].item(), 3)
        })
    return rezultate

user_inp = input("Input: ")
top_recomandari = recomanda_prajituri(user_input_clean)

print("\nTop recomandari:")
for r in top_recomandari:
    print(f"{r['denumire']} -> scor similaritate: {r['scor']}")


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



Top recomandari:
Tartaletta cu fistic -> scor similaritate: 0.505
Prajitura Andreea -> scor similaritate: 0.491
Minichoux cu rodie -> scor similaritate: 0.482


Curatare text

In [20]:
import re
import unidecode

def clean_text(text):
    text = text.lower()
    text = unidecode.unidecode(text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

Codificarea input-ului si output-ului

In [21]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

#Model DistilBERT multilingual
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
model = AutoModel.from_pretrained("distilbert-base-multilingual-cased")

def embed_text(texts):
    #Returneaza embedding pentru o lista de texte
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embeddings = model_output.last_hidden_state[:,0,:]
    return F.normalize(embeddings, p=2, dim=1)


Incarcare model spaCy pentru extragere keywords

In [29]:
import re

# stopwords minimale în română (poți extinde lista)
STOPWORDS_RO = {
    "cu", "si", "sau", "de", "la", "in", "pe", "din", "care", "pentru",
    "un", "o", "al", "ai", "ale", "alege", "este", "este", "ca"
}

def extract_keywords(text, min_len=3):
    """
    Extrage cuvinte-cheie simple din text:
    - elimină stopwords
    - păstrează doar cuvinte mai lungi de min_len
    """
    words = re.findall(r"\b\w+\b", text.lower())
    keywords = [w for w in words if w not in STOPWORDS_RO and len(w) >= min_len]
    return keywords


In [31]:
def recomanda_prajituri(user_input, top_k=3):
    user_input_clean = clean_text(user_input)

    #Extrage termeni-cheie
    keywords = extract_keywords(user_input_clean)

    #Filtrare prăjituri care conțin toți termenii-cheie
    if keywords:
        mask = df["descriere_clean"].apply(lambda x: all(k in x for k in keywords))
        df_filtered = df[mask] if mask.any() else df  # fallback = toate prăjiturile
    else:
        df_filtered = df

    #Embeddinguri pentru subset
    vecs_filtered = embed_text(df_filtered["descriere_clean"].tolist())
    user_vec = embed_text([user_input_clean])

    #Similaritate cosinus
    cos_scores = torch.matmul(user_vec, vecs_filtered.T)[0]  # (num_docs,)
    top_idx = torch.topk(cos_scores, k=min(top_k, len(df_filtered))).indices.tolist()

    #Construiește lista de rezultate
    rezultate = []
    for i in top_idx:
        idx_df = df_filtered.index[i]
        rezultate.append({
            "denumire": df_filtered["denumire"].iloc[i],
            "scor": round(cos_scores[i].item(), 3)
        })
    return rezultate

user_inp = input("Scrie ce tip de desert vrei: ")
top_recomandari = recomanda_prajituri(user_inp)

print("\nTop recomandari:")
for r in top_recomandari:
    print(f"{r['denumire']} -> scor similaritate: {r['scor']}")




Top recomandari:
Negresă de post -> scor similaritate: 0.975
Piure de castane -> scor similaritate: 0.974
Mini Carrot cake  -> scor similaritate: 0.974


In [46]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

model = SentenceTransformer('distiluse-base-multilingual-cased-v1')
df = pd.read_csv("../Prajituri.csv")

#descrierea prajiturilor
def create_text(row):
    return f"{row['denumire']}: contine {row['ingrediente']}. Textura: {row['Textura']}, gust: {row['Dulceata']}, ingredient de baza: {row['Ingredient de baza']}"

df["descriere"] = df.apply(create_text, axis=1)

#embeddinguri
desert_embeddings = model.encode(df["descriere"].tolist(), convert_to_tensor=True)

test_inputs = [
    "Vreau ceva dulce cu ciocolată și vișine",
    "Poftesc o prăjitură crocantă cu fistic",
    "Aș mânca un desert cu cremă de lămâie și biscuiți",
    "Vreau o prăjitură cu brânză și zmeură",
    "Îmi plac deserturile cu ciocolată amăruie și rom"
]

for input_text in test_inputs:
    print(f"\nInput: {input_text}")
    input_embedding = model.encode(input_text, convert_to_tensor=True)
    cosine_scores = util.cos_sim(input_embedding, desert_embeddings)[0]
    top_results = torch.topk(cosine_scores, k=3)

    for score, idx in zip(top_results[0], top_results[1]):
        idx = idx.item()
        print(f"  {df.iloc[idx]['denumire']} (scor: {score:.4f})")




Input: Vreau ceva dulce cu ciocolată și vișine
  Tort Cherry Dream (scor: 0.3635)
  Mini cupe ciocolată, cremă whisky și caramel (scor: 0.3081)
  Minibaba (scor: 0.3014)

Input: Poftesc o prăjitură crocantă cu fistic
  Tartaletta cu fistic (scor: 0.2619)
  Minichoux cu fistic (scor: 0.2495)
  Tort Chocodrop pistachio (scor: 0.2211)

Input: Aș mânca un desert cu cremă de lămâie și biscuiți
  Mini Mousse caramel și măr (scor: 0.3488)
  Monoporție Ciocolată albă cu caise (scor: 0.3367)
  Mousse ciocolată și zmeură  (scor: 0.3320)

Input: Vreau o prăjitură cu brânză și zmeură
  Monoporție Ricotta (scor: 0.2642)
  Monoporție Pavlova (scor: 0.2386)
  Mini Carrot cake  (scor: 0.2287)

Input: Îmi plac deserturile cu ciocolată amăruie și rom
  Monoporție Pădurea Neagră  (scor: 0.3337)
  Mousse ciocolată și zmeură  (scor: 0.3299)
  Monoporție Ricotta (scor: 0.3141)


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import pandas as pd
import numpy as np

# Presupunem ca 'df' si 'model' sunt deja incarcate din codul tau anterior
# Asigura-te ca in CSV ai coloana 'Ingredient de baza' curata (ex: Ciocolata, Fructe, Branza)

# 1. Definim un set de date de TEST (Ground Truth)
# Aici pui propozitii pe care le-ar scrie un user si categoria reala pe care o astepti
test_data = [
    ("Vreau o prajitura cu multa ciocolata", "Ciocolata"),
    ("Ceva racoritor cu fructe de padure", "Fructe"),
    ("Ador prajiturile cu branza dulce", "Branza"),
    ("O amandina insiropata", "Ciocolata"),
    ("Tarta cu capsuni si vanilie", "Fructe"),
    ("Ecler cu ness si crema", "Lapte/Oua"), # Sau alta categorie din CSV-ul tau
    ("Prajitura cu mere de post", "Fructe")
]

true_labels = []
predicted_labels = []

print("Rulare teste pentru matricea de confuzie...")

for text, expected_category in test_data:
    # 2. Facem predictia folosind logica ta existenta
    # Folosim functia ta de recomandare sau logica directa
    user_vec = model.encode(text, convert_to_tensor=True)

    # Calculam similaritatea cu toate prajiturile
    # Nota: cakes_vecs trebuie sa fie deja calculat pe tot dataframe-ul
    if 'cakes_vecs' not in locals():
         cakes_vecs = model.encode(df["descriere"].tolist(), convert_to_tensor=True)

    cos_scores = util.cos_sim(user_vec, cakes_vecs)[0]

    # Luam doar primul rezultat (Top 1)
    best_match_idx = np.argmax(cos_scores.cpu().numpy())

    # Extragem categoria (Ingredient de baza) a prajiturii recomandate
    predicted_category = df.iloc[best_match_idx]["Ingredient de baza"]

    true_labels.append(expected_category)
    predicted_labels.append(predicted_category)

# 3. Calculam Acuratetea
acc = accuracy_score(true_labels, predicted_labels)
print(f"Acuratete pe setul de test: {acc * 100:.2f}%")

# 4. Generam Matricea de Confuzie
labels = sorted(list(set(true_labels) | set(predicted_labels)))
cm = confusion_matrix(true_labels, predicted_labels, labels=labels)

# 5. Desenam Graficul (Heatmap)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Ce a prezis Modelul (Ingredientul prajiturii recomandate)')
plt.ylabel('Ce a cerut Utilizatorul (Intentia)')
plt.title('Matrice de Confuzie - Recomandare Prăjituri')
plt.show()

Rulare teste pentru matricea de confuzie...


NameError: name 'model' is not defined

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Pregatim datele
# Luam embeddingurile generate (cakes_vecs)
# Convertim in numpy array daca sunt tensor
X = cakes_vecs.cpu().numpy()

# 2. Reducem dimensionalitatea la 2D folosind t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5) # Perplexity mic pt seturi mici de date
X_embedded = tsne.fit_transform(X)

# 3. Cream un DataFrame pentru plotare
df_plot = df.copy()
df_plot['x'] = X_embedded[:, 0]
df_plot['y'] = X_embedded[:, 1]

# 4. Desenam graficul
plt.figure(figsize=(12, 10))
sns.scatterplot(
    data=df_plot,
    x='x',
    y='y',
    hue='Ingredient de baza', # Punctele vor fi colorate dupa ingredient
    style='Textura',          # Forma punctului dupa textura (optional)
    s=100,                    # Marimea punctelor
    palette='viridis'
)

# Optional: Adaugam etichete (numele prajiturilor) pe grafic
# Atentie: Daca ai multe prajituri, textul se va suprapune
for i in range(df_plot.shape[0]):
    plt.text(
        df_plot.x[i]+0.2,
        df_plot.y[i],
        df_plot.denumire[i],
        fontsize=9
    )

plt.title("Vizualizarea Spațiului Semantic al Prăjiturilor (t-SNE)")
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()